# 스트림 방식으로 출력하기 p 222

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
llm = ChatOpenAI(model="gpt-4o-mini")

llm.invoke([HumanMessage("잘 지냈어?")])

AIMessage(content='네, 잘 지내고 있습니다! 당신은 어떻게 지내셨어요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 12, 'total_tokens': 29, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'chatcmpl-C3ZHSNvopn01zEpo07mff3kzDzHeC', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--f545bfdf-d7b3-4edb-9c53-b7ceb81e41f4-0', usage_metadata={'input_tokens': 12, 'output_tokens': 17, 'total_tokens': 29, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [2]:
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool # @tool 데코레이터를 사용하여 함수를 도구로 등록
def get_current_time(timezone: str, location: str) -> str:
    """ 현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time

In [3]:
# 도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
llm_with_tools = llm.bind_tools(tools)

In [4]:
from langchain_core.messages import SystemMessage

# (4) 사용자의 질문과 tools 사용하여 llm 답변 생성
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

# (5) llm_with_tools를 사용하여 사용자의 질문에 대한 llm 답변 생성
response = llm_with_tools.invoke(messages)
messages.append(response)

# (6) 생성된 llm 답변 출력
print(messages)

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_HXk0SGJ3UyRAwQ14gsIPDmxQ', 'function': {'arguments': '{"timezone":"Asia/Seoul","location":"부산"}', 'name': 'get_current_time'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 135, 'total_tokens': 158, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_62a23a81ef', 'id': 'chatcmpl-C3ZHhOiKedHF0kkTF8hevmPa4qW4b', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--9ae4dcdc-06cf-46cb-b2dd-bfca4245e41b-0', tool_calls

In [5]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
    print(tool_call["args"]) # (8) 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # (9) 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '부산'}
Asia/Seoul (부산) 현재시각 2025-08-12 11:44:29 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_HXk0SGJ3UyRAwQ14gsIPDmxQ', 'function': {'arguments': '{"timezone":"Asia/Seoul","location":"부산"}', 'name': 'get_current_time'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 135, 'total_tokens': 158, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_62a23a81ef', 'id': 'chatcmpl-C3ZHhOiKedHF0kkTF8hevmPa4qW4b', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--9ae4dcdc-06cf-46cb-b2dd-bfca4245e41b-0', tool_cal

In [6]:
llm_with_tools.invoke(messages)


AIMessage(content='부산은 현재 2025년 8월 12일 11시 44분입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 192, 'total_tokens': 215, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_62a23a81ef', 'id': 'chatcmpl-C3ZI09Hi5hk7LUwxvKWsqKHVYJTnP', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--c2bc0d6e-16c0-4f74-b39e-1a019f42f31e-0', usage_metadata={'input_tokens': 192, 'output_tokens': 23, 'total_tokens': 215, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [7]:
from pydantic import BaseModel, Field

class StockHistoryInput(BaseModel):
    ticker: str = Field(..., title="주식 코드", description="주식 코드 (예: AAPL)")
    period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d, 1mo, 1y)")

In [8]:
import yfinance as yf

@tool
def get_yf_stock_history(stock_history_input: StockHistoryInput) -> str:
    """ 주식 종목의 가격 데이터를 조회하는 함수"""
    stock = yf.Ticker(stock_history_input.ticker)
    history = stock.history(period=stock_history_input.period)
    history_md = history.to_markdown() 

    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)

In [9]:
messages.append(HumanMessage("테슬라는 한달 전에 비해 주가가 올랐나 내렸나?"))

response = llm_with_tools.invoke(messages)
print(response)
messages.append(response)

content='' additional_kwargs={'tool_calls': [{'id': 'call_YGRIIzp83BO0oYkmflnHrQKl', 'function': {'arguments': '{"stock_history_input":{"ticker":"TSLA","period":"1mo"}}', 'name': 'get_yf_stock_history'}, 'type': 'function'}], 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 283, 'total_tokens': 310, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'chatcmpl-C3ZITE7YztnbSfikX3WzYd9GjS3XO', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='run--cd872397-122f-4125-9303-021bfb49b919-0' tool_calls=[{'name': 'get_yf_stock_history', 'args': {'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}, 'id': 'call_YGRIIzp83BO0oYkmflnHrQKl', 'type': 'tool_call'}] usage_metadata={'inp

In [10]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    print(tool_msg)

{'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}
content='| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |\n|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|\n| 2025-07-14 00:00:00-04:00 | 317.73 | 322.6  | 312.67 |  316.9  | 7.80434e+07 |           0 |              0 |\n| 2025-07-15 00:00:00-04:00 | 319.68 | 321.2  | 310.5  |  310.78 | 7.75563e+07 |           0 |              0 |\n| 2025-07-16 00:00:00-04:00 | 312.8  | 323.5  | 312.62 |  321.67 | 9.72848e+07 |           0 |              0 |\n| 2025-07-17 00:00:00-04:00 | 323.15 | 324.34 | 317.06 |  319.41 | 7.39229e+07 |           0 |              0 |\n| 2025-07-18 00:00:00-04:00 | 321.66 | 330.9  | 321.42 |  329.65 | 9.4255e+07  |           0 |              0 |\n| 2025-07-21 00:00:00-04:00 | 334.4  | 338    | 326.88 |  328.49 | 7.57688e+07 |           0 |              0 |\n| 2025-07-22 00:00:00-04:0

In [11]:
llm_with_tools.invoke(messages)

AIMessage(content='테슬라(TSLA)의 주가는 한 달 전에 비해 올랐습니다. \n\n- 한 달 전(2025년 7월 11일)의 종가는 approximately 316.90달러였습니다.\n- 현재(2025년 8월 11일)의 종가는 approximately 339.03달러입니다.\n\n따라서 주가는 약 22.13달러 상승했습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 1585, 'total_tokens': 1671, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'chatcmpl-C3ZIfIcvcghYc2NlrWAyu2BkdewQz', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--18704de8-33d3-4c73-888b-af202f4b6c64-0', usage_metadata={'input_tokens': 1585, 'output_tokens': 86, 'total_tokens': 1671, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## 08-4 스트림 출력

In [12]:
for c in llm.stream([HumanMessage("잘 지냈어? 한국 사회의 문제점에 대해 이야기해줘.")]):
    print(c.content, end='|') 

|안|녕하세요|!| 잘| 지|냈|습니다|.| 한국| 사회|의| 문제|점|에| 대해| 이야기|해|보|면|,| 여러| 가지|가| 있습니다|.| 몇| 가지| 주요| 문제|점을| 살|펴|보|겠습니다|.

|1|.| **|저|출|산| 및| 고|령|화|**|:| 한국|은| 세계|에서| 가장| 낮|은| 출|산|율|을| 기록|하고| 있으며|,| 이에| 따라| 고|령|화| 문제|도| 심|각|합니다|.| 젊|은| 세|대|의| 경제|적| 부담|이| 커|지고|,| 노동|력| 부족|과| 사회| 안전|망|에| 대한| 압|박|이| 증가|하고| 있습니다|.

|2|.| **|주|택| 문제|**|:| 대|도|시|,| 특히| 서울|의| 주|택| 가격|이| sky|-high|로| 상승|하면서| 주|거| 안정|성이| 크게| 위|협|받|고| 있습니다|.| 많은| 젊|은| 층|이| 적|절|한| 가격|의| 주|택|을| 구|하는| 데| 어려|움을| 겪|고| 있으며|,| 이는| 사회|적| 불|만|을| 초|래|하고| 있습니다|.

|3|.| **|직|장| 문화|와| 비|정|규|직| 문제|**|:| 한국|의| 직|장| 문화|는| 종|종| 장|시간| 노동|과| 높은| 경쟁|을| 강조|합니다|.| 비|정|규|직|의| 비|율|이| 높|아|지|면서| 고|용|의| 안정|성이| 감소|하고|,| 이는| 경제|적| 불|안|정을| 야|기|합니다|.

|4|.| **|학|벌|주의|**|:| 여|전히| 한국| 사회|에서는| 학|벌|이| 중요한| 평가| 기준|으로| 작|용|하고| 있습니다|.| 이|로| 인해| 개인|의| 능|력|이나| 노력|보다| 출|신|학교|에| 따라| 기|회|가| 제한|되는| 문제가| 있습니다|.

|5|.| **|사회| 양|극|화|**|:| 빈|부| 격|차|가| 확대|되|면서| 사회|적| 갈|등|이| 심|화|되고| 있습니다|.| 경제|적| 불|균|형|은| 교육|,| 건강|,| 주|거| 등| 다양한| 분야|에서| 불|평|등|을| 초|래|하고| 있습니다|.

|6|.| **|정|신| 건강| 문제|**|:| 정신

In [13]:
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

response = llm_with_tools.stream(messages)

# 파편화된 tool_call 청크를 하나로 합치기 
is_first = True
for chunk in response:    
    print("chunk type: ", type(chunk))
    
    if is_first:
        is_first = False
        gathered = chunk
    else:
        gathered += chunk
    
    print("content: ", gathered.content, "tool_call_chunk", gathered.tool_calls)

messages.append(gathered)

chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_zb5PqjxqvaorZTUNF4t1oWmR', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_zb5PqjxqvaorZTUNF4t1oWmR', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_zb5PqjxqvaorZTUNF4t1oWmR', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {'timezone': ''}, 'id': 'call_zb5PqjxqvaorZTUNF4t1oWmR', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {'timezone': 'Asia'}, 'id': 'call_zb5PqjxqvaorZTUNF4t1oWmR', 'type': 'tool_c

In [14]:
gathered

AIMessageChunk(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_zb5PqjxqvaorZTUNF4t1oWmR', 'function': {'arguments': '{"timezone":"Asia/Seoul","location":"부산"}', 'name': 'get_current_time'}, 'type': 'function'}]}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_34a54ae93c', 'service_tier': 'default'}, id='run--ce000241-cc51-4d4d-add1-956531d3184a', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_zb5PqjxqvaorZTUNF4t1oWmR', 'type': 'tool_call'}], tool_call_chunks=[{'name': 'get_current_time', 'args': '{"timezone":"Asia/Seoul","location":"부산"}', 'id': 'call_zb5PqjxqvaorZTUNF4t1oWmR', 'index': 0, 'type': 'tool_call_chunk'}])

In [15]:
for tool_call in gathered.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # tool_dict를 사용하여 도구 이름으로 도구 함수를 선택
    print(tool_call["args"]) # 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '부산'}
Asia/Seoul (부산) 현재시각 2025-08-12 11:46:10 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessageChunk(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_zb5PqjxqvaorZTUNF4t1oWmR', 'function': {'arguments': '{"timezone":"Asia/Seoul","location":"부산"}', 'name': 'get_current_time'}, 'type': 'function'}]}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_34a54ae93c', 'service_tier': 'default'}, id='run--ce000241-cc51-4d4d-add1-956531d3184a', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_zb5PqjxqvaorZTUNF4t1oWmR', 'type': 'tool_call'}], tool_call_chunks=[{'name': 'get_current_time', 'args': '{"timezone":"Asia/Seoul","location":"부산"}', 'id': 'call_zb5PqjxqvaorZTUNF4t1oWmR', 'index': 0, 'type': 'tool_call_chunk'}]),
 ToolMessage(content='Asia/S

In [16]:
for c in llm_with_tools.stream(messages):
    print(c.content, end='|')

|부|산|은| 지금| |202|5|년| |8|월| |12|일| |11|시| |46|분|입니다|.||